In [1]:
import torch
import pandas as pd
import numpy as np

from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments
)

from sklearn.metrics import classification_report, accuracy_score
from sklearn.model_selection import train_test_split
from imblearn.over_sampling import RandomOverSampler

In [2]:
MODEL_CKPT = "bert-base-uncased"
dataset_name = "dair-ai/emotion"

In [5]:
id2label = {
    0: "sadness",
    1: "happy",
    2: "love",
    3: "anger",
    4: "fear",
    5: "surprise"
}

label2id = {v: k for k, v in id2label.items()}
TARGET_LABELS = list(id2label.values())
NUM_LABELS = len(TARGET_LABELS)

In [7]:
print(f"Loading {dataset_name}...")
dataset = load_dataset(dataset_name)

Loading dair-ai/emotion...


In [9]:
df = pd.DataFrame(dataset["train"])

# Map joy → happy
LABEL_REMAP = {
    "sadness": "sadness",
    "joy": "happy",
    "love": "love",
    "anger": "anger",
    "fear": "fear",
    "surprise": "surprise"
}

df["label"] = df["label"].map(
    lambda x: label2id[LABEL_REMAP[dataset["train"].features["label"].names[x]]]
)

In [11]:
# 4. Balance Dataset
# =====================================
X = df["text"].values.reshape(-1, 1)
y = df["label"].values

ros = RandomOverSampler(random_state=42)
X_resampled, y_resampled = ros.fit_resample(X, y)

balanced_df = pd.DataFrame({
    "text": X_resampled.flatten(),
    "label": y_resampled
})

print("Balanced label distribution:")
print(balanced_df["label"].value_counts())


Balanced label distribution:
label
0    5362
3    5362
2    5362
5    5362
4    5362
1    5362
Name: count, dtype: int64


In [13]:
# ✅ REQUIRED LINE
df = balanced_df

print("Balanced label distribution:")
print(df["label"].value_counts())

Balanced label distribution:
label
0    5362
3    5362
2    5362
5    5362
4    5362
1    5362
Name: count, dtype: int64


In [23]:
# Initialize Tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_CKPT)

def preprocess_function(examples):
    # Max length 128 is usually sufficient for tweets
    return tokenizer(examples["text"], truncation=True, max_length=128)

print("Tokenizing dataset...")
encoded_dataset = dataset.map(preprocess_function, batched=True)

Tokenizing dataset...


Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

In [27]:
from peft import get_peft_model, LoraConfig, TaskType
from sklearn.metrics import accuracy_score, classification_report
print(f"\nLoading {MODEL_CKPT} and applying LoRA...")

# 1. Load Base Model
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_CKPT,
    num_labels=len(TARGET_LABELS),
    id2label=id2label,
    label2id=label2id
)

# 2. Define LoRA Configuration
peft_config = LoraConfig(
    task_type=TaskType.SEQ_CLS, # Sequence Classification
    r=16,                       # Rank (8 or 16 are common)
    lora_alpha=32,              # Alpha (usually 2x rank)
    lora_dropout=0.1,
    bias="none",
    target_modules=["query", "value"] # Apply to BERT attention layers
)


Loading bert-base-uncased and applying LoRA...


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [29]:
# 3. Wrap Model with PEFT
model = get_peft_model(model, peft_config)
model.print_trainable_parameters()

trainable params: 594,438 || all params: 110,081,292 || trainable%: 0.5400


In [31]:
args = TrainingArguments(
    output_dir="./bert-base-lora-dair-emotion",
    eval_strategy="epoch",      # Evaluate every epoch
    save_strategy="epoch",      # Save checkpoint every epoch
    learning_rate=2e-4,         # Higher LR is often better for LoRA
    per_device_train_batch_size=32, 
    per_device_eval_batch_size=32,
    num_train_epochs=5,         # 5 epochs is usually enough for this dataset
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    fp16=torch.cuda.is_available(), # Use mixed precision if GPU available
    logging_dir='./logs',
    report_to="none"
)

In [33]:
def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=1)
    acc = accuracy_score(labels, predictions)
    return {"accuracy": acc}

In [35]:
from transformers import Trainer

# 1. Define a Custom Trainer to handle the compatibility issue
class CustomTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        # The default Trainer adds 'num_items_in_batch' to inputs, which causes the error.
        # We override this to call the model directly without that argument.
        outputs = model(**inputs)
        
        # Extract the loss
        loss = outputs.get("loss") if isinstance(outputs, dict) else outputs[0]
        
        return (loss, outputs) if return_outputs else loss

In [39]:
from transformers import (
    AutoTokenizer, 
    AutoModelForSequenceClassification, 
    TrainingArguments, 
    Trainer, 
    DataCollatorWithPadding
)

In [41]:
trainer = CustomTrainer(
    model=model,
    args=args,
    train_dataset=encoded_dataset["train"],
    eval_dataset=encoded_dataset["validation"],
    processing_class=tokenizer,
    data_collator=DataCollatorWithPadding(tokenizer=tokenizer),
    compute_metrics=compute_metrics,
)

In [43]:
# 3. Start Training
print("\nStarting Training with CustomTrainer...")
trainer.train()


Starting Training with CustomTrainer...


C:\Users\vijay\anaconda3\Lib\site-packages\transformers\models\bert\modeling_bert.py:413: UserWarning: 1Torch was not compiled with flash attention. (Triggered internally at ..\aten\src\ATen\native\transformers\cuda\sdp_utils.cpp:263.)
  attn_output = torch.nn.functional.scaled_dot_product_attention(


Epoch,Training Loss,Validation Loss,Accuracy
1,1.007300,0.515509,0.817500
2,0.416100,0.264282,0.914500
3,0.261900,0.203869,0.931000
4,0.210700,0.193095,0.933000
5,0.185100,0.177613,0.937500


TrainOutput(global_step=2500, training_loss=0.41623036499023436, metrics={'train_runtime': 280.0134, 'train_samples_per_second': 285.701, 'train_steps_per_second': 8.928, 'total_flos': 2157062893443072.0, 'train_loss': 0.41623036499023436, 'epoch': 5.0})

In [45]:
print("\nEvaluating on Test Set...")
test_results = trainer.predict(encoded_dataset["test"])


Evaluating on Test Set...


In [47]:
y_preds = np.argmax(test_results.predictions, axis=1)
y_true = test_results.label_ids

print(f"Test Accuracy: {accuracy_score(y_true, y_preds):.4f}")
print("\nClassification Report (Note: 'joy' is now labeled 'happy'):")
print(classification_report(y_true, y_preds, target_names=TARGET_LABELS))

Test Accuracy: 0.9280

Classification Report (Note: 'joy' is now labeled 'happy'):
              precision    recall  f1-score   support

     sadness       0.96      0.96      0.96       581
       happy       0.95      0.95      0.95       695
        love       0.84      0.82      0.83       159
       anger       0.92      0.92      0.92       275
        fear       0.89      0.93      0.91       224
    surprise       0.81      0.70      0.75        66

    accuracy                           0.93      2000
   macro avg       0.89      0.88      0.89      2000
weighted avg       0.93      0.93      0.93      2000



In [51]:
# Save the model
model.save_pretrained("./bert_lora_happy_model_2")
print("Model saved!")

Model saved!


In [53]:
import gradio as gr
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from peft import PeftModel, PeftConfig

# ==========================================
# 1. SETUP
# ==========================================
# Path where we saved the model in the previous step
adapter_path = "./bert_lora_happy_model"
base_model_name = "bert-base-uncased"

# Define the label mapping (Must match training!)
id2label = {
    0: 'sadness', 
    1: 'happy', 
    2: 'love', 
    3: 'anger', 
    4: 'fear', 
    5: 'surprise'
}
label2id = {v: k for k, v in id2label.items()}

# Device setup
device = "cuda" if torch.cuda.is_available() else "cpu"

# ==========================================
# 2. LOAD MODEL & TOKENIZER
# ==========================================
print("Loading model components...")

# Load Tokenizer
tokenizer = AutoTokenizer.from_pretrained(base_model_name)

# Load Base Model
base_model = AutoModelForSequenceClassification.from_pretrained(
    base_model_name,
    num_labels=len(id2label),
    id2label=id2label,
    label2id=label2id
)

# Load the LoRA Adapter onto the base model
model = PeftModel.from_pretrained(base_model, adapter_path)
model.to(device)
model.eval()

print("Model loaded successfully!")

# ==========================================
# 3. DEFINE PREDICTION FUNCTION
# ==========================================
def predict_emotion(text):
    if not text:
        return None
        
    # Tokenize
    inputs = tokenizer(
        text, 
        return_tensors="pt", 
        truncation=True, 
        max_length=128
    ).to(device)
    
    # Predict
    with torch.no_grad():
        outputs = model(**inputs)
        logits = outputs.logits
        # Convert logits to probabilities
        probs = torch.nn.functional.softmax(logits, dim=1)
    
    # Format output for Gradio (Label: Confidence dict)
    # Convert tensor to python list
    scores = probs[0].tolist()
    
    # Create dictionary {Label: Score}
    results = {id2label[i]: score for i, score in enumerate(scores)}
    return results

# ==========================================
# 4. LAUNCH GRADIO INTERFACE
# ==========================================
demo = gr.Interface(
    fn=predict_emotion,
    inputs=gr.Textbox(
        lines=2, 
        placeholder="Type a sentence here (e.g., 'I am so proud of you!')",
        label="Input Text"
    ),
    outputs=gr.Label(num_top_classes=3, label="Predicted Emotions"),
    title="😊 Emotion Detection with BERT + LoRA",
    description="Enter a sentence to detect its emotion. The model was trained to recognize: **Sadness, Happy, Love, Anger, Fear, and Surprise**.",
    examples=[
        ["I finally got the promotion I wanted!"],
        ["I feel so alone in this big city."],
        ["Why did you lie to me?"],
        ["The sudden noise scared me to death."],
        ["I love watching the sunset with you."]
    ],
    theme="default"
)

# Launch the app
demo.launch(share=True)

Loading model components...


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Model loaded successfully!
* Running on local URL:  http://127.0.0.1:7860

Could not create share link. Please check your internet connection or our status page: https://status.gradio.app.
